In [161]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
sns.set_style('whitegrid')
sns.set_palette("Set2")

%matplotlib inline

# Leer los datos

In [162]:
totales = pd.read_csv("../../../data/respuestas_fede.csv")
print(totales.shape)

#globales
marmol = totales.loc[totales["escuela"] == "Colegio Modelo Mármol"]
mantovani = totales.loc[totales["escuela"] == "Escuela Nueva Juan Mantovani"]

file_ext = '.png'
dpi_value = 200

(369, 22)


## Qué es la nube vs Funcionalidad de la nube

In [163]:
# funcion aux para analizar misc
def analizarMiscFuncionalidad(val, agrupando = False):
    lista = val.split(",")
    misc = 0
    not_misc = 0
    if(lista == ["0"]): # Si no seleccionaron ninguna rta, es porque contestaron no se en la pregunta anterior
        return "Sin\nRta." 
    
    # respuestas sin misconception, # not_misc
    if "Lanubemepermiteguardarlasfotosyvideosdelcelular" in lista:
        not_misc = not_misc + 1
    if "InstagramyTikTokusanlanubeparacompartirsusvideos" in lista:
        not_misc = not_misc + 1
    if "Sepuedeusarlanubeparajugarjuegossininstalarlos" in lista:
        not_misc = not_misc + 1
    if "GoogleMapsdescargasusmapasdelanube" in lista:
        not_misc = not_misc + 1

    # respuestas con misconception, # misc
    if "PodemosutilizarlanubesinconexiónaInternet" in lista:
        misc = misc + 1
    if "Sinlanubeseríaimposibleescucharmúsicaenelcelular" in lista:
        misc = misc + 1
    if "Sinlanubenoseríaposiblehacerllamadasporcelular" in lista:
        misc = misc + 1
    if "Sinlanubeseríaimposiblesacarfotosconelcelular" in lista:
        misc = misc + 1
    
    if(not agrupando):
        if(not_misc == misc):
            return "Iguales\nMisc. y\nNo Misc."    
        if(misc == 0):
            return "Ninguna\nMisc."        
        if(not_misc == 0):
            return "Todas\nMisc."      
        if(not_misc > misc):        
            return "Minoría\nMisc."    
        if(misc > not_misc):
            return "Mayoría\nMisc."
    else:
        if(not_misc == misc):
            return "Iguales\nMisc. y\nNo Misc."      
        if(not_misc > misc):        
            return "Minoría\nMisc."    
        if(misc > not_misc):
            return "Mayoría\nMisc."

def analizarMiscNube(val):
    if val == "Una nube":
        return "Misc. fuerte"
    if val == "Una parte del celular":
        return "Misc. fuerte"
    if val == "Una computadora gigante":
        return "Misc. fuerte"
    if val == "Una red de antenas y satélites":
        return "Misc. débil"
    if val == "Muchísimas computadoras (tantas que podrían llenar una cancha de fútbol)":
        return "Sin Misc."
    

In [164]:
# Boilerplate para generar grafico de nube vs func de nube
def nube_vs_funcionalidad(df, df_name, agrupando = False, invertir_porcentajes = False):
    que_es_nube = df["que_es_nube"]
    que_es_nube_sin_nose = que_es_nube[que_es_nube != "No sé"]

    funcionalidad_nube = df["afirmaciones_nube"].str.replace(" ", "").fillna("0").apply(lambda x: analizarMiscFuncionalidad(x,agrupando))
    funcionalidad_nube_sin_sinrta = funcionalidad_nube[funcionalidad_nube != "Sin\nRta."]

    if(agrupando):
        labels_que_es_nube_sin_nose = [
                                    "Misc. fuerte",
                                    "Misc. débil",
                                    "Sin Misc.",
                                    ]
        labels_funcionalidad_nube_sin_sinrta = [
                    "Minoría\nMisc.",
                    "Iguales\nMisc. y\nNo Misc.",
                    "Mayoría\nMisc.",
                   ]
        
        que_es_nube_sin_nose = que_es_nube_sin_nose.apply(analizarMiscNube)

        if(not invertir_porcentajes):
            filas    = labels_que_es_nube_sin_nose
            columnas = labels_funcionalidad_nube_sin_sinrta

            cross_tab_result = pd.crosstab(que_es_nube_sin_nose, funcionalidad_nube_sin_sinrta)
        else:
            filas    = labels_funcionalidad_nube_sin_sinrta
            filas.reverse()
            columnas = labels_que_es_nube_sin_nose
            columnas.reverse()

            cross_tab_result = pd.crosstab(funcionalidad_nube_sin_sinrta, que_es_nube_sin_nose)
        
    else:
        labels_que_es_nube_sin_nose = [
                 "Una nube",
                 "Una parte\ndel celu",
                 "Una compu\ngigante",
                 "Una red\nde antenas\ny satélites",
                 "Muchísimas\ncompus",
                ]
        labels_funcionalidad_nube_sin_sinrta = [
                    "Ninguna\nMisc.",
                    "Minoría\nMisc.",
                    "Iguales\nMisc. y\nNo Misc.",
                    "Mayoría\nMisc.",
                    "Todas\nMisc.",
                   ]

        rename_que_es_nube_sin_nose = {"Muchísimas computadoras (tantas que podrían llenar una cancha de fútbol)": "Muchísimas\ncompus",
                                "Una computadora gigante":"Una compu\ngigante",
                                "Una parte del celular": "Una parte\ndel celu",
                                "Una red de antenas y satélites": "Una red\nde antenas\ny satélites"}
        
        if(not invertir_porcentajes):
            filas = labels_que_es_nube_sin_nose
            columnas = labels_funcionalidad_nube_sin_sinrta
            cross_tab_result = pd.crosstab(que_es_nube_sin_nose, funcionalidad_nube_sin_sinrta)
            cross_tab_result = cross_tab_result.rename(index=rename_que_es_nube_sin_nose)
        else:
            filas = labels_funcionalidad_nube_sin_sinrta
            filas.reverse()
            columnas = labels_que_es_nube_sin_nose
            columnas.reverse()
            cross_tab_result = pd.crosstab(funcionalidad_nube_sin_sinrta, que_es_nube_sin_nose)
            cross_tab_result = cross_tab_result.rename(columns=rename_que_es_nube_sin_nose)

    cross_tab_result = cross_tab_result.reindex(filas, columns=columnas)

    def agregar_totales(fila):
       res = 0
       for i in list(range(0,fila.size)):
            res = res + fila[i]
       return res
    
    def dividir(fila):
       return fila.div(fila[fila.size - 1])

    cross_tab_result['totales'] = cross_tab_result.apply(agregar_totales, axis=1)
    cross_tab_result = cross_tab_result.apply(dividir, axis=1)
    cross_tab_result = cross_tab_result.drop(['totales'], axis=1)

    titulo_que_es_nube_sin_nose = '¿Que es la nube?'
    titulo_funcionalidad_nube = 'Misconceptions en funcionalidad de la nube'

    if(not invertir_porcentajes):
        titulo_filas = titulo_que_es_nube_sin_nose
        titulo_columnas = titulo_funcionalidad_nube
    else:
        titulo_filas = titulo_funcionalidad_nube
        titulo_columnas = titulo_que_es_nube_sin_nose

    sns.heatmap(cross_tab_result, cmap='YlGnBu', annot=True, fmt='.2f', linewidths=0.5, vmin=0.0, vmax=1.0)
    plt.title('Relación de Respuestas\nQue es la nube vs Funcionalidad de la nube\nDatos {} - Agrupando {} - Invirtiendo {}'.format(df_name,agrupando,invertir_porcentajes))
    plt.xlabel(titulo_columnas)
    plt.xticks(rotation=0)
    plt.ylabel(titulo_filas)
    plt.yticks(rotation=0)
    # plt.show()
    plt.savefig('nube_vs_func_{}_agrupando_{}_invirtiendo_{}'.format(df_name,agrupando,invertir_porcentajes)+file_ext, bbox_inches='tight', dpi=dpi_value)
    plt.clf()


In [187]:
# Boilerplate para generar grafico de nube vs func de nube
def nube_vs_funcionalidad_alternativo(df, df_name, invertir_porcentajes = False):
    que_es_nube = df["que_es_nube"]
    que_es_nube_sin_nose = que_es_nube[que_es_nube != "No sé"]
    # print(que_es_nube_sin_nose == "Muchísimas computadoras (tantas que podrían llenar una cancha de fútbol)")

    funcionalidad_nube= totales["afirmaciones_nube"].str.replace(" ", "").fillna("0")
    funcionalidad_nube_sin_sinrta = funcionalidad_nube[funcionalidad_nube != "0"]
    funcionalidad_nube_sin_sinrta = funcionalidad_nube_sin_sinrta.str.get_dummies(sep=',')
    # print(funcionalidad_nube_sin_sinrta)

    datas = []

    for col_name in funcionalidad_nube_sin_sinrta.columns:
        d = funcionalidad_nube_sin_sinrta[funcionalidad_nube_sin_sinrta[col_name] == 1]
        d = d[col_name]
        if(not invertir_porcentajes):
            d = pd.crosstab(que_es_nube_sin_nose, d).rename(columns={1:col_name})
        else:
            d = pd.crosstab(d,que_es_nube_sin_nose).rename(index={1:col_name})
        datas.append(d)

    # print(datas)

    rename_que_es_nube = {"Muchísimas computadoras (tantas que podrían llenar una cancha de fútbol)": "Muchísimas\ncompus",
                    "Una computadora gigante":"Una compu\ngigante",
                    "Una parte del celular": "Una parte\ndel celu",
                    "Una red de antenas y satélites": "Una red\nde antenas\ny satélites"}

    rename_funcionalidad_nube = {"Lanubemepermiteguardarlasfotosyvideosdelcelular": "Guardar\n fotos\n y videos\n del celu",
                            "InstagramyTikTokusanlanubeparacompartirsusvideos": "Apps\nVideos",
                            "Sepuedeusarlanubeparajugarjuegossininstalarlos": "Juegos\nsin\ninstalar",
                            "GoogleMapsdescargasusmapasdelanube": "Google\n Maps",
                            "PodemosutilizarlanubesinconexiónaInternet": "Uso\nsin\nInternet",
                            "Sinlanubeseríaimposibleescucharmúsicaenelcelular": "Req.\nmúsica",
                            "Sinlanubenoseríaposiblehacerllamadasporcelular": "Req.\nllamadas",
                            "Sinlanubeseríaimposiblesacarfotosconelcelular": "Req.\nfotos\ncelu"
                            }

    if(not invertir_porcentajes):
        cross_tab_result = pd.concat(datas, axis=1).fillna(0)
        # print(cross_tab_result)
        cross_tab_result = cross_tab_result.rename(index=rename_que_es_nube,
                    columns=rename_funcionalidad_nube)
    else:
        cross_tab_result = pd.concat(datas, axis=0).fillna(0)
        cross_tab_result = cross_tab_result.rename(index=rename_funcionalidad_nube,
                    columns=rename_que_es_nube)

    labels_que_es_nube = [  "Una nube",
                            "Una parte\ndel celu",
                            "Una compu\ngigante",
                            "Una red\nde antenas\ny satélites",
                            "Muchísimas\ncompus",]
    labels_funcionalidad_nube = [   "Guardar\n fotos\n y videos\n del celu",
                                    "Apps\nVideos",
                                    "Juegos\nsin\ninstalar",
                                    "Google\n Maps",
                                    "Uso\nsin\nInternet",
                                    "Req.\nmúsica",
                                    "Req.\nllamadas",
                                    "Req.\nfotos\ncelu"]

    titulo_nube ='¿Que es la nube?'
    titulo_funciones ='Funciones nube'

    if (not invertir_porcentajes):
        titulo_columnas = titulo_funciones
        titulo_filas = titulo_nube
        cross_tab_result = cross_tab_result.reindex(index=labels_que_es_nube, columns=labels_funcionalidad_nube)
    else:
        titulo_columnas = titulo_nube
        titulo_filas = titulo_funciones
        labels_funcionalidad_nube.reverse()
        labels_que_es_nube.reverse()
        cross_tab_result = cross_tab_result.reindex(index=labels_funcionalidad_nube, columns=labels_que_es_nube)

    def agregar_totales(fila):
       res = 0
       for i in list(range(0,fila.size)):
            res = res + fila[i]
       return res
    
    def dividir(fila):
       return fila.div(fila[fila.size - 1])

    cross_tab_result['totales'] = cross_tab_result.apply(agregar_totales, axis=1)
    cross_tab_result = cross_tab_result.apply(dividir, axis=1)
    cross_tab_result = cross_tab_result.drop(['totales'], axis=1)

    sns.heatmap(cross_tab_result, cmap='YlGnBu', annot=True, fmt='.2f', linewidths=0.5, vmin=0.0, vmax=1.0)
    plt.title('Alternativo\nRelación de Respuestas\nQue es la nube vs Funcionalidad de la nube\nDatos {} - Invirtiendo {}'.format(df_name,invertir_porcentajes))
    plt.xlabel(titulo_columnas)
    plt.xticks(rotation=0)
    plt.ylabel(titulo_filas)
    plt.yticks(rotation=0)
    # plt.show()
    plt.savefig('alternativo_nube_vs_func_{}_invirtiendo_{}'.format(df_name,invertir_porcentajes)+file_ext, bbox_inches='tight', dpi=dpi_value)
    plt.clf()


In [166]:
# llamo al generador de graficos con los tres dfs
nube_vs_funcionalidad(totales, "totales")
nube_vs_funcionalidad(marmol, "marmol")
nube_vs_funcionalidad(mantovani, "mantovani")

# Agrupando misc
nube_vs_funcionalidad(totales,   "totales", True)
nube_vs_funcionalidad(marmol,    "marmol", True)
nube_vs_funcionalidad(mantovani, "mantovani", True)

# Invirtiendo
nube_vs_funcionalidad(totales,   "totales",   invertir_porcentajes=True)
nube_vs_funcionalidad(marmol,    "marmol",    invertir_porcentajes=True)
nube_vs_funcionalidad(mantovani, "mantovani", invertir_porcentajes=True)

# Agrupando misc e Invirtiendo
nube_vs_funcionalidad(totales,   "totales",  agrupando=True, invertir_porcentajes=True)
nube_vs_funcionalidad(marmol,    "marmol",   agrupando=True, invertir_porcentajes=True)
nube_vs_funcionalidad(mantovani, "mantovani",agrupando=True, invertir_porcentajes=True)


<Figure size 640x480 with 0 Axes>

In [188]:
#### Alternativos ############
# llamo al generador de graficos con los tres dfs
nube_vs_funcionalidad_alternativo(totales,   "totales")
nube_vs_funcionalidad_alternativo(marmol,    "marmol")
nube_vs_funcionalidad_alternativo(mantovani, "mantovani")

# Invirtiendo
nube_vs_funcionalidad_alternativo(totales,   "totales",   invertir_porcentajes=True)
nube_vs_funcionalidad_alternativo(marmol,    "marmol",    invertir_porcentajes=True)
nube_vs_funcionalidad_alternativo(mantovani, "mantovani", invertir_porcentajes=True)

<Figure size 640x480 with 0 Axes>